[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kartikmunjal/rlhf-and-reward-modelling/blob/main/notebooks/01_data_exploration.ipynb)

# Notebook 1 — Exploring the Anthropic hh-rlhf Dataset

Before writing a single line of training code, we need to understand the data that drives the entire pipeline.  
Everything downstream — the SFT model, the reward model, and the RL policy — is shaped by this dataset.

**What is hh-rlhf?**  
Anthropic's `hh-rlhf` (helpful & harmless RLHF) dataset contains ~170k human preference annotations.  
For each item, a human was shown two model responses to the same prompt and marked which one was better.  
The result is a *preference pair*: a `chosen` response (preferred) and a `rejected` response (dispreferred).

**Why does the data format matter?**  
The Bradley-Terry reward model we train in Stage 2 is only as good as the annotation signal here.  
Understanding length distributions, conversation structure, and what makes a response "better" directly
informs our preprocessing choices and model evaluation strategy.

In [ ]:
# Install dependencies (Colab)
# !pip install -q datasets transformers matplotlib seaborn

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from datasets import load_dataset
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
from collections import Counter
import re

print('Loading hh-rlhf harmless-base split...')
dataset = load_dataset('Anthropic/hh-rlhf', data_dir='harmless-base')
print(dataset)

## Dataset Structure

Each example contains two fields:
- **`chosen`** — the full conversation (Human/Assistant turns) ending with the *preferred* response
- **`rejected`** — the same conversation up to the final Assistant turn, which is *dispreferred*

Crucially, the `chosen` and `rejected` texts share the same prompt (all Human/Assistant turns except the last).
Only the final Assistant response differs between them.

In [ ]:
# Inspect one example
ex = dataset['train'][42]
print('=== CHOSEN (preferred response) ===')
print(ex['chosen'][:800])
print()
print('=== REJECTED (dispreferred response) ===')
print(ex['rejected'][:800])

## Parsing Prompts and Responses

For reward model training we need to split each conversation into:
- **Prompt** — all turns up to and including the final `\n\nAssistant:` marker
- **Response** — only the final assistant turn

This separation matters because the reward model scores the *full* sequence (prompt + response),
but we only want it to evaluate the quality of the *response*, not re-penalise the prompt.

In [ ]:
from src.data.preprocessing import extract_prompt_and_response

for i in range(3):
    ex = dataset['train'][i]
    prompt, response = extract_prompt_and_response(ex['chosen'])
    print(f'--- Example {i} ---')
    print(f'PROMPT (last 200 chars): ...{prompt[-200:]}')
    print(f'RESPONSE: {response[:300]}')
    print()

## Length Distributions

Understanding sequence lengths is critical for two reasons:
1. GPT-2 has a hard 1024-token limit — we need to know how much truncation occurs at `max_length=512`
2. Chosen vs rejected length difference can reveal annotation artifacts
   (e.g., if chosen responses are systematically longer, the reward model might learn length ≈ quality)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

sample = dataset['train'].select(range(2000))

chosen_lens, rejected_lens, response_lens = [], [], []
for ex in sample:
    chosen_lens.append(len(tokenizer.encode(ex['chosen'])))
    rejected_lens.append(len(tokenizer.encode(ex['rejected'])))
    _, resp = extract_prompt_and_response(ex['chosen'])
    response_lens.append(len(tokenizer.encode(resp)))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, data, title, color in zip(
    axes,
    [chosen_lens, rejected_lens, response_lens],
    ['Chosen (full)', 'Rejected (full)', 'Response only (chosen)'],
    ['steelblue', 'tomato', 'seagreen'],
):
    ax.hist(data, bins=50, color=color, edgecolor='white', linewidth=0.5)
    ax.axvline(512, color='black', linestyle='--', linewidth=1.5, label='max_length=512')
    ax.set_title(title)
    ax.set_xlabel('Token count')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.text(0.98, 0.95, f'median={int(np.median(data))}',
            transform=ax.transAxes, ha='right', va='top', fontsize=9)

plt.suptitle('Token length distributions (n=2000)', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig('length_distributions.png', dpi=100, bbox_inches='tight')
plt.show()

trunc_pct = sum(1 for l in chosen_lens if l > 512) / len(chosen_lens) * 100
print(f'Truncation rate at max_length=512: {trunc_pct:.1f}% of examples')
print(f'Chosen mean length: {np.mean(chosen_lens):.0f} tokens (std={np.std(chosen_lens):.0f})')
print(f'Rejected mean length: {np.mean(rejected_lens):.0f} tokens (std={np.std(rejected_lens):.0f})')

## Chosen vs Rejected: Length Bias

A key question: do annotators simply prefer longer responses?  
If so, the reward model will learn a length heuristic, not quality.  
The scatter plot below checks for this correlation.

In [ ]:
chosen_resp_lens, rejected_resp_lens = [], []
for ex in sample:
    _, c_resp = extract_prompt_and_response(ex['chosen'])
    _, r_resp = extract_prompt_and_response(ex['rejected'])
    chosen_resp_lens.append(len(tokenizer.encode(c_resp)))
    rejected_resp_lens.append(len(tokenizer.encode(r_resp)))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scatter: chosen vs rejected length
axes[0].scatter(rejected_resp_lens[:500], chosen_resp_lens[:500],
                alpha=0.3, s=10, color='steelblue')
m = max(max(rejected_resp_lens[:500]), max(chosen_resp_lens[:500]))
axes[0].plot([0, m], [0, m], 'k--', linewidth=1, label='equal length')
axes[0].set_xlabel('Rejected response length (tokens)')
axes[0].set_ylabel('Chosen response length (tokens)')
axes[0].set_title('Chosen vs Rejected response lengths')
axes[0].legend()

# Length difference distribution
diffs = [c - r for c, r in zip(chosen_resp_lens, rejected_resp_lens)]
axes[1].hist(diffs, bins=50, color='mediumpurple', edgecolor='white')
axes[1].axvline(0, color='black', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('chosen_length - rejected_length (tokens)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Length difference: chosen minus rejected')
axes[1].text(0.02, 0.95, f'mean diff = {np.mean(diffs):.1f}',
             transform=axes[1].transAxes, va='top', fontsize=9)

plt.tight_layout()
plt.show()

chosen_longer = sum(1 for d in diffs if d > 0)
print(f'Chosen longer than rejected: {chosen_longer/len(diffs)*100:.1f}% of pairs')
print('Mild length bias exists — but responses with chosen shorter also appear, so it is not purely length-driven.')

## Key Takeaways

1. **~30–40% of full conversations exceed 512 tokens** — truncation is expected; we drop trailing tokens
2. **Mild length bias**: chosen responses are on average ~20 tokens longer, but the distribution strongly overlaps — quality signals exist beyond length
3. **Response-only lengths (median ~80 tokens)** fit comfortably within the 512-token budget
4. **Conversation format is consistent**: always `Human: ... \n\nAssistant: ...` — our `extract_prompt_and_response` parser handles all examples correctly

---
**Next**: [02_sft_training.ipynb](02_sft_training.ipynb) — Fine-tune GPT-2 on the chosen responses